In [1]:
#!pip install langchain langchain-community langchain-core transformers==4.52.4
#!pip install fastapi uvicorn pyngrok accelerate -q

### **Importing Libraries**

In [2]:
from fastapi import FastAPI, Request, HTTPException, UploadFile, File
import uvicorn, threading, time, socket, tempfile, re
from pyngrok import ngrok, conf
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate
from langchain.document_loaders import PyPDFLoader

### **Loading Model**

In [3]:
model_name = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.float16,device_map="auto")

2026-01-10 22:39:17.725069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768084757.744397     216 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768084757.750744     216 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768084757.767827     216 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768084757.767846     216 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768084757.767849     216 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### **Text Generation**

In [4]:
def generate_text(prompt, max_length=4096, num_return_sequences=1):
    """Generate text with better parameters"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,  # Use max_new_tokens instead of max_length
        num_return_sequences=num_return_sequences,
        do_sample=False,  # Deterministic output
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.1,  # Low temperature for more focused output
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove the input prompt from the output
    if generated_text.startswith(prompt[:100]):  # Check if prompt is in output
        # Try to remove it
        prompt_end = generated_text.find(format_instructions[:50])
        if prompt_end != -1:
            # Find where actual response starts (after format instructions)
            actual_start = generated_text.find("{", prompt_end)
            if actual_start != -1:
                generated_text = generated_text[actual_start:]
    
    return generated_text


### **Defining Schemas**

In [5]:
full_name_schema = ResponseSchema(name="full_name",description="The full name of the person in the CV")
email_schema = ResponseSchema(name="email",description="The email address of the person")
education_schema = ResponseSchema(name="education",description="The education details such as degree, institution, and graduation year")
skills_schema = ResponseSchema(name="skills",description="A list of technical or soft skills mentioned in the CV")
experience_schema = ResponseSchema(name="experience",description="Work experience details including role, company, and years")

response_schemas = [full_name_schema, email_schema, education_schema, skills_schema, experience_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

### **CV Template**

In [6]:
CV_extraction_template = """You are an intelligent CV parser. Extract information from the resume and return ONLY a valid JSON object.

Extract these fields:
- full_name: The person's complete name
- email: Email address
- education: Education details (degree, institution, year)
- skills: Comma-separated list of skills
- experience: Work experience (role, company, years)

{format_instructions}

Resume text:
{resume_text}

Respond with ONLY the JSON object. No explanation, no markdown formatting, just the JSON."""


### **Extract JSON block**

In [7]:
import re
import json
def extract_json_block(text):
    """Extract and validate JSON from model output"""
    
    # Method 1: Try to find JSON in markdown code blocks
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    
    if matches:
        json_str = matches[-1].strip()
    else:
        # Method 2: Try to find raw JSON (look for outermost { ... })
        pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
        matches = re.findall(pattern, text, re.DOTALL)
        if matches:
            # Get the longest match (most likely to be complete)
            json_str = max(matches, key=len)
        else:
            # Method 3: Last resort - extract everything between first { and last }
            start = text.find('{')
            end = text.rfind('}')
            if start != -1 and end != -1:
                json_str = text[start:end+1]
            else:
                return None
    
    # Clean up common issues
    json_str = json_str.strip()
    
    # Try to validate and fix
    try:
        # Validate it's proper JSON
        parsed = json.loads(json_str)
        return json_str
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print(f"Problematic JSON: {json_str[:200]}...")
        
        # Try to fix common issues
        try:
            # Remove trailing commas before } or ]
            fixed = re.sub(r',(\s*[}\]])', r'\1', json_str)
            parsed = json.loads(fixed)
            print("Fixed JSON by removing trailing commas")
            return fixed
        except:
            print("Could not auto-fix JSON")
            return None


In [8]:
NGROK_TOKEN = "37qVnS46FzKpnz7zXUuo7ERf2lI_7VYnYgG87T7gNnyX2U8aA"
API_KEY = "secret123"

In [12]:
app = FastAPI()
@app.post("/parse_cv")
async def parse_cv(file: UploadFile = File(...), req: Request = None):
    """Parse CV and extract structured information"""
    
    # Authentication
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    # Save uploaded file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(await file.read())
        pdf_path = tmp.name

    try:
        # Load PDF
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        resume_text = " ".join([page.page_content for page in pages])
        
        # Limit text length to avoid token limits
        if len(resume_text) > 4000:
            resume_text = resume_text[:4000]

        # Create prompt
        prompt = PromptTemplate(
            template=CV_extraction_template,
            input_variables=["resume_text", "format_instructions"],
        ).format(resume_text=resume_text, format_instructions=format_instructions)

        print("Generating response from model...")
        raw_output = generate_text(prompt)
        
        print(f"Raw model output (first 500 chars): {raw_output[:500]}")
        
        # Extract and validate JSON
        json_output = extract_json_block(raw_output)
        
        if json_output is None:
            print(f"Failed to extract JSON. Full output: {raw_output}")
            raise HTTPException(
                status_code=500, 
                detail={
                    "error": "Failed to extract valid JSON from model output",
                    "raw_output": raw_output[:1000]
                }
            )
        
        # Validate it's proper JSON
        try:
            json.loads(json_output)
        except json.JSONDecodeError as e:
            raise HTTPException(
                status_code=500,
                detail={
                    "error": f"Invalid JSON: {str(e)}",
                    "json_output": json_output[:500]
                }
            )
        
        return {"parsed_cv": json_output, "status": "success"}
        
    except Exception as e:
        print(f"Error in parse_cv: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        # Cleanup temp file
        try:
            import os
            os.unlink(pdf_path)
        except:
            pass

In [13]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()

conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://leptophyllous-cachectic-colleen.ngrok-free.dev


INFO:     Started server process [216]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:35769 (Press CTRL+C to quit)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response from model...
Raw model output (first 500 chars): {
	"full_name": string  // The full name of the person in the CV
	"email": string  // The email address of the person
	"education": string  // The education details such as degree, institution, and graduation year
	"skills": string  // A list of technical or soft skills mentioned in the CV
	"experience": string  // Work experience details including role, company, and years
}
```

Resume text:
Gasser Ahmed Mohamed
AI Engineer
gasserahmed478@gmail.com
 
+20 1270006496, +201553928106
 
Giza, Egypt

INFO:     196.132.100.28:0 - "POST /parse_cv HTTP/1.1" 200 OK


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response from model...
Raw model output (first 500 chars): {
	"full_name": string  // The full name of the person in the CV
	"email": string  // The email address of the person
	"education": string  // The education details such as degree, institution, and graduation year
	"skills": string  // A list of technical or soft skills mentioned in the CV
	"experience": string  // Work experience details including role, company, and years
}
```

Resume text:
Gasser Ahmed Mohamed
AI Engineer
gasserahmed478@gmail.com
 
+20 1270006496, +201553928106
 
Giza, Egypt

INFO:     196.132.100.28:0 - "POST /parse_cv HTTP/1.1" 200 OK


...

Generating response from model...
Raw model output (first 500 chars): {
	"full_name": string  // The full name of the person in the CV
	"email": string  // The email address of the person
	"education": string  // The education details such as degree, institution, and graduation year
	"skills": string  // A list of technical or soft skills mentioned in the CV
	"experience": string  // Work experience details including role, company, and years
}
```

Resume text:
Gasser Ahmed Mohamed
AI Engineer
gasserahmed478@gmail.com
 
+20 1270006496, +201553928106
 
Giza, Egypt

INFO:     196.132.100.28:0 - "POST /parse_cv HTTP/1.1" 200 OK


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response from model...
Raw model output (first 500 chars): {
	"full_name": string  // The full name of the person in the CV
	"email": string  // The email address of the person
	"education": string  // The education details such as degree, institution, and graduation year
	"skills": string  // A list of technical or soft skills mentioned in the CV
	"experience": string  // Work experience details including role, company, and years
}
```

Resume text:
Gasser Ahmed Mohamed
AI Engineer
gasserahmed478@gmail.com
 
+20 1270006496, +201553928106
 
Giza, Egypt

INFO:     196.132.100.28:0 - "POST /parse_cv HTTP/1.1" 200 OK


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response from model...
Raw model output (first 500 chars): {
	"full_name": string  // The full name of the person in the CV
	"email": string  // The email address of the person
	"education": string  // The education details such as degree, institution, and graduation year
	"skills": string  // A list of technical or soft skills mentioned in the CV
	"experience": string  // Work experience details including role, company, and years
}
```

Resume text:
Gasser Ahmed Mohamed
AI Engineer
gasserahmed478@gmail.com
 
+20 1270006496, +201553928106
 
Giza, Egypt

INFO:     196.132.100.28:0 - "POST /parse_cv HTTP/1.1" 200 OK
